# Image & Signal Processing
Here, we'll demonstrate how you could implement visualization & processing techniques for biological time series and images using Python.

![](Data/leukaemia_smear.jpeg)

### By the end of this notebook, you'll be able to:
* Choose an appropriate sampling rate for time series
* Plot time series data with accurate timestamps
* Filter 2D and 3D data using convolution
* Display & manipulate images in Python
* Apply different types of filters to images

<hr>

First, let's get setup...

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.image as mpimg # New module to read in images


...and define some plotting functions here to save space later. You can collapse this cell by clicking **View > Collapse Selected Code**

In [ ]:
def get_kspace_data(path='Data/T2-MRI-Image.jpeg'):
    '''Generates a dictionary that contains kspace data. 
    Keys: Full, Undersampled-x, Undersampled-y, Highpass'''
    img = mpimg.imread(path)
    img = img.mean(axis=2)

    kspace_data = {}

    kspace = np.fft.fft2(img)
    kspace_shifted = np.fft.fftshift(kspace)
    
    kspace_data['Full'] = kspace
    
    kspace_data['Undersampled-x'] = kspace_shifted.copy()
    kspace_data['Undersampled-x'][:,::2] = 0
    kspace_data['Undersampled-x'] = np.fft.ifftshift(kspace_data['Undersampled-x'])
    
    kspace_data['Undersampled-y'] = kspace_shifted.copy()
    kspace_data['Undersampled-y'][::2,:] = 0
    kspace_data['Undersampled-y'] = np.fft.ifftshift(kspace_data['Undersampled-y'])

    
    kspace_data['Highpass'] = kspace_shifted.copy()
    kspace_data['Highpass'][225:275,225:275] = 0
    kspace_data['Highpass'] = np.fft.ifftshift(kspace_data['Highpass'])


    return kspace_data
    
def plot_kspace(kspace, ax=None, title="k-space (log magnitude)",cmap='gray'):
    # Shift zero-frequency component to center for visualization
    kspace_shifted = np.fft.fftshift(kspace)

    # Magnitude for display
    kspace_display = np.log(np.abs(kspace_shifted) + 1)

    if ax is None:
        # If no axes provided, create a new figure
        fig, ax = plt.subplots()

    ax.imshow(kspace_display, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')

def transform_and_plot(kspace, ax=None, title=None, cmap='gray'):
    recon = np.fft.ifft2(kspace)
    recon = np.real(recon)

    if ax is None:
        # If no axes provided, create a new figure
        fig, ax = plt.subplots()

    ax.imshow(recon, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')

def plot_sampled_signal(fs_new, time,
                        wave_frequency=10):
    wave_frequency = wave_frequency
    sin_wave = np.sin(2 * np.pi * wave_frequency * time)
    
    plt.plot(time, sin_wave)
    
    sampled_signal = np.rint(np.linspace(0, len(time)-1, fs_new)).astype(int)
    
    
    plt.plot(time[sampled_signal], sin_wave[sampled_signal],'--o')
    #plt.plot(sample_times, sample_sin, '-o')
    plt.title('True Waveform vs Sampled Wave')
    plt.xlabel('time (ms)')
    plt.ylabel('Amplitude')
    plt.legend(['True Waveform', 'Sampling Result'],loc='best', bbox_to_anchor=(0.7, 0., 0.7, 0.7))
    
    print("Sampling Frequency / Wave Frequency = ", fs_new / wave_frequency)
    plt.show()

## The Nyquist Criterion and Aliasing
When the sampling rate is too slow, this results in poor reconstruction of the sampled signal, and high-frequency components of the signal are misidentified. This type of sampling artifact is known as "aliasing." To prevent aliasing, a sufficiently fast sampling rate must be chosen.

**Below we'll explore how changing the sampling rate alters the reconstructed signal.**

<div class="alert alert-success">

**Task**: Change the variable `fs_new` to alter the sampling frequency. Try a few different values. What do you notice?

</div>

In [ ]:
#Generate a wave
T = 1 #period (s)
timepoints = 1000 #sampling frequency (Hz)
time = np.linspace(0, T, T * timepoints) #time vector (ms)

#Choose a new sampling frequency
fs_new = ...

#plot the result
plot_sampled_signal(fs_new, time)

The **Nyquist Criterion** tells us that to recover our signal, we must sample our signal *at least twice as fast* as the fastest frequency it contains. When our sampling frequency doesn't meet this threshold, high-frequency information is lost and we recover a wave containing slower frequencies than our original signal!

Even after our sampling rate passes the Nyquist criterion, recovered waveforms can have altered amplitudes, so we generally try to sample as quickly as possible  

## Time Series Signal Processing

To practice working with time-series data, we'll import a recording of slow-wave sleep from a young individual, collected by the Walker Lab at UC Berkeley. This data was collected at 100 Hz from channel 'F3'. This sampling frequency is fine for EEG data, but wouldn't be enough for high frequency spiking data. That kind of data is typically sampled at 40 **kilo**Hz.

In [ ]:
# Load the .txt file
full_recording = np.loadtxt('Data/recording.txt')
print(full_recording.shape)
full_recording

Wow! This is a pretty big data file. 

It contains **1,032,000** samples, but let's try to figure out how long that is in seconds. Since the sampling frequency of the data is 100 Hz, this means that each sample is taken every $\frac{1}{100}$ seconds.

We can create a vector of timepoints for the whole dataset by using the NumPy function <code>np.arange()</code> over the length of our full recording and then multiplying by the time it takes to record each sample.

<div class="alert alert-success">

**Task**: Create a `time_vector` variable so that your data is plotted in seconds, not samples.

</div>

In [ ]:
# Define sampling frequency, num_samples, and time vector
sampling_freq = 100 # sampling frequency, in Hz
time_step = ...
time_vector = np.arange(...) * time_step
# ADD CODE HERE
time_vector

<div class="alert alert-success">

**Task**: Plot your signal! You only need to add a line of code where it says `# ADD CODE HERE`

</div>

In [ ]:
# Plot the signal
fig, ax = plt.subplots(figsize=(12, 4))

# ADD CODE HERE

plt.xlabel('Time (seconds)')
plt.ylabel('Voltage')
plt.title('N3 sleep EEG data (F3)')
plt.show()

The full recording is 10000 seconds-- that's nearly 3 hours! There are also some extremely large voltage changes early in the recording and at the end that we probably don't want in our analysis, so we'll need to select a clean section of our data to analyze.

Let's slice out a more manageable, 30-second chunk of this data. 
1. Start your slice around index <code>300000</code>
2. Find your stop index by adding $30 s * 100 Hz$ to your start index
3. Calculate <code>time_step = 1 / sampling_freq</code>
4. Use <code>np.arange(start=start_idx, stop=stop_idx, step=time_step)

In [ ]:
data = ...
time_vector = np.arange(...)
plt.plot(...)
plt.xlabel('Time (seconds)')
plt.ylabel('Voltage')
plt.title('N3 sleep EEG data (F3): 30-s window')
plt.show()

### Filtering a signal with convolution
Let's start with a simple step signal (left) which we'll add some noise to (right).

In [ ]:
step_signal = np.zeros(100) # Create a 1D array of all 0s
step_signal[50:] = 1        # Set 50: to 1

fig, ax = plt.subplots(1,2,figsize=(10,4))
ax[0].plot(step_signal)

# Create random signal
np.random.seed(0) # Set the "seed" for randomness

noisy_signal = (step_signal + np.random.normal(0, 0.35, step_signal.shape))
ax[1].plot(noisy_signal)

plt.show()

If our goal is to recover our original signal (like the image on the left) from a noisy measurement (like the image on the right), we can **smooth** the signal. In the first line below, we'll take each point and average it by one datapoint to the left and one datapoint to the right, discarding the ends of the signal, where we can't do this operation.

In [ ]:
# Take the mean of neighboring datapoints
smooth_signal = (noisy_signal[:-1] + noisy_signal[1:]) / 2.0

plt.plot(smooth_signal)
plt.show()

Above, we're only averaging by one datapoint on each side. What if we average by 3 datapoints? Below, we'll do the following:
1. Create an output array called `smooth_signal3`, of the same length as noisy_signal.
2. At each element in `smooth_signal3` starting at point 1, and ending at point -2, place the average of the sum of: 1/3 of the element to the left of it in noisy_signal, 1/3 of the element at the same position, and 1/3 of the element to the right.
3. Discard the leftmost and rightmost elements.

In [ ]:
# Generate our signal
smooth_signal3 = (noisy_signal[:-2] + noisy_signal[1:-1] + noisy_signal[2:]) / 3

plt.plot(smooth_signal, label='mean of 2')
plt.plot(smooth_signal3, label='mean of 3')
plt.legend(loc='upper left')
plt.show()

If we want to continue this, the first line of our code is going to continue to get longer, and there isn't a straightforward way to make this flexible for how many points we want to average. Thankfully, there's a name for what we're doing: **convolving**. This same concept, nearest-neighbor averages, can be expressed as a **convolution with an averaging kernel**.

**Convolution** is the process of adding each element of the image to its local neighbors, weighted by the kernel.

In [ ]:
# Same as above, using a convolution kernel
# Neighboring pixels multiplied by 1/3 and summed

# Create the "kernel". For 3, this creates an array that looks like [0.333,0.333,0.333]
mean_kernel3 = np.full((3,), 1/3)

# Smooth the signal by convolving it
smooth_signal3p = np.convolve(noisy_signal, mean_kernel3, mode='valid')

plt.plot(smooth_signal3p)
plt.show()

In [ ]:
# Check that they're equal!
print('smooth_signal3 and smooth_signal3p are equal:',
      np.allclose(smooth_signal3, smooth_signal3p))

<div class="alert alert-success">

**Tasks**:
    
1. What happens if we convolve with a kernel of size 10? Also, do these arrays have the same # of data points? If not, why?
2. Smooth our sleep recording with different kernel sizes and inspect.
   
</div>

In [ ]:
np.convolve?

In [ ]:
# Try a kernel of size 10 here

# Create the "kernel". For 3, this creates an array that looks like [0.333,0.333,0.333]
mean_kernel10 = ...

# Smooth the signal by convolving it
smooth_signal10p = ...

plt.plot(smooth_signal10p)
plt.show()

When we perform convolution with <code>mode='valid'</code> this changes the length of our array. If we want our output to have the same length as our input signal, we'll have to use <code>mode='same'</code>.

Let's use this approach to smooth our 30-s chunk of EEG data using <code>mean_kernel10</code>. We'll plot our original signal and our smoothed signal using subplots!

In [ ]:
# Smooth the recording we imported above!
smooth_data = ...
fig, ax = plt.subplots(2,1)

ax[0].plot(time_vector,data)
ax[1].plot(time_vector, smooth_data)

ax[0].set_title("Original Data")
ax[1].set_title("Smoothed Data")

ax[0].set_xlabel("Time (s)")
ax[1].set_xlabel("Time (s)")

ax[0].set_ylabel("Voltage")
ax[1].set_ylabel("Voltage")

fig.tight_layout()

## MR Reconstruction

Signals measured by MRI systems are acquired as phase and frequency information in **K-space** before being converted to an image using the Fourier transform.

Let's load in some K-space data and store it in a dictionary called <code>kspace_data</code>. K-space data can be accessed from the dictionary using the following keys: <code>'Full'</code>, <code>'Undersampled-x'</code>, <code>'Undersampled-y'</code>, and <code>'Highpass'</code>

In [ ]:
kspace_data = get_kspace_data()

Just like with time-series, we have to be careful when sampling in 2D. Below we'll plot K-space images that demonstrate:
1. Undersampling in the x-direction
2. Undersampling in the y-direction
3. Removal of low frequencies
4. Full K-space

What do you notice about these plots? How do you think this will change the reconstructed image?

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(10,8))
fig.suptitle("K-space Plots")
plot_kspace(kspace_data['Undersampled-x'],title='Undersampled (x)', ax=ax[0,0])
plot_kspace(kspace_data['Undersampled-y'],title='Undersampled (y)', ax=ax[0,1])
plot_kspace(kspace_data['Highpass'],title='No Low Frequencies', ax=ax[1,0])
plot_kspace(kspace_data['Full'],title='Full Kspace', ax=ax[1,1])
fig.tight_layout()

Let's apply the Fourier Transform to our K-space data to try to recover the scanned object! We can use our K-space data in our custom function <code>transform_and_plot()</code> just like we did above with <code>plot_kspace()</code>. Give it a try in the code block below!

<div class="alert alert-success">

**Task**: Plot the transformed data below and note the effects of different sampling approaches
   
</div>

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(10,8))
fig.suptitle("Reconstructed Plots")
transform_and_plot(...,title='Undersampled (x)', ax=ax[0,0])
transform_and_plot(...,title='Undersampled (y)', ax=ax[0,1])
transform_and_plot(...,title='No Low Frequencies', ax=ax[1,0])
transform_and_plot(...,title='Full Kspace', ax=ax[1,1])
fig.tight_layout()

## Images in Python
Below, we'll use `mpimg.imread()` to read in an image file and [`plt.imshow()`](https://matplotlib.org/devdocs/api/_as_gen/matplotlib.pyplot.imshow.html) to show [our image](https://en.wikipedia.org/wiki/Cancer_cell#/media/File:Acute_lymphoblastic_leukaemia_smear.jpg).

In [ ]:
leukaemia_img = 'Data/leukaemia_smear.jpeg'
cells = mpimg.imread(leukaemia_img)
plt.axis('off') # Turn axis ticks & labels off
plt.imshow(cells)
plt.show()

We can work with our `cells` object just like we would any other object. Let's check its shape.

In [ ]:
# Check cells shape
type(cells)
cells.shape

**If the first and second value are the size, what is the third value?**


<div class="alert alert-success">

**Task** Create a variable called `cells_gray` that consists of only the values in the third (Blue) channel of the image

In [ ]:
cells_gray = ...
print(cells_gray)
plt.imshow(cells_gray)
plt.colorbar(shrink=0.75)
plt.show()

By default, matplotlib uses the viridis color map to plot a sequence of values (you can see the luminance at each pixel by showing the colorbar with `plt.colorbar()`).

<div class="alert alert-success">

**Task** Change your image to a different colormamp using the `cmap` argument in `plt.imshow` so that you get a Black & White image.
</div>

In [ ]:
# ADD CODE HERE
plt.imshow(cells_gray, ...)
plt.colorbar(shrink=0.75)
plt.show()

<div class= "alert alert-success">

**Task** Using the [documentation](https://matplotlib.org/stable/users/explain/colors/colormaps.html) for choosing colormaps, choose three colormaps for the variable `cmap_list` that corresponds to each of the color channels in `cells`.

</div>

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

# Create subplots with colorbars
cmap_list = ...
for i in range(3):
    im = ax[i].imshow(cells[:, :, i], cmap=cmap_list[i])
    fig.colorbar(im, ax=ax[i], shrink=0.5)  # Add colorbar to each subplot
    ax[i].set_title(f'Channel {i+1}')

plt.tight_layout()
plt.show()

### Filtering images to smooth them
In our signal processing tutorial above, we talked about convolving a signal with an averaging kernel to smooth the signal. We can apply similar logic to images. Let's start with a very simple image.

In [ ]:
bright_square = np.zeros((7, 7), dtype=float)
bright_square[2:5, 2:5] = 1
bright_square

This gives the values above, and looks like this!

In [ ]:
plt.imshow(bright_square, cmap = "gray")
plt.axis('off')
plt.show()

For our first example of a filter, consider the following filtering array, which we’ll call a “mean kernel”. For each pixel, a kernel defines which neighboring pixels to consider when filtering, and how much to weight those pixels.

In [ ]:
mean_kernel = np.full((3, 3), 1/9)

print(mean_kernel)

Now, let’s take our mean kernel and apply it to every pixel of the image.

Applying a (linear) filter essentially means:

1. Center a kernel on a pixel

2. Multiply the pixels under that kernel by the values in the kernel

3. Sum all the those results

4. Replace the center pixel with the summed result

This process is known as **convolution** (the same process we applied to a signal, but now in 3D!)

In [ ]:
# Import image processing toolbox
import scipy.ndimage as ndi

# Set precision
%precision 2

print(ndi.convolve(bright_square, mean_kernel))

In [ ]:
smooth_square = ndi.convolve(bright_square, mean_kernel)
plt.imshow(smooth_square,cmap='gray')
plt.show()

This kind of mean filter is a bit brute. Typically, we'd use a Gaussian filter, where the amount of filtering depends on the distance from the center point. Thankfully, we can do that using the filters module:

In [ ]:
from skimage import filters

# Apply Gaussian smoothing
sigma = 3
gaussian_cells = filters.gaussian(cells_gray,sigma)

# Plot
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(image, cmap='gray')
plt.title("Original")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(smoothed, cmap='gray')
plt.title(f"Gaussian σ={sigma}")
plt.axis('off')
plt.show()

### Edge filtering
In biology, we often want to filter edges to identify different features within cells, or cells within a piece of tissue. To do so, we'll often use some sort of edge filtering. To give you an intuition for how edge filtering works, first consider taking our step function and convolving it with a filter that is [-1 0 1].

**Note**: For technical signal processing reasons, convolutions actually occur “back to front” between the input array and the kernel which produces a negative signal spike at the edge. **Correlations** are like convolutions but occur in the signal order, so we’ll use correlate below to visualize a positive spike in the plot below.

In [ ]:
# Recreate the step signal
step_signal = np.zeros(100)
step_signal[50:] = 1

# Convolve it with a [-1 0 1] filter
edge_filter = np.correlate(step_signal, np.array([-1, 0, 1]),mode='valid')

plt.plot(step_signal, label='signal')
plt.plot(edge_filter, linestyle='dashed', label='correlated')
plt.legend(loc='upper left')
plt.show()

Whenever neighboring values are close, the filter response is close to 0. Right at the boundary of a step, we’re subtracting a small value from a large value and and get a spike in the response. This spike “identifies” our edge.

Now, let's apply this to images. Below, we'll just apply one filter at a time. in the vertical direction. The vertical kernel has already been created for you.

To ensure that our pixel values remain in a valid range, we'll first convert our image dtype from <code>uint8</code> to <code>float</code> by using the <code>.astype(float)</code> method on our image array. If we don't do this, the filters won't behave properly.

<div class="alert alert-success">

**Task** Create a horizontal kernel and assign it to the `horizontal_kernel` variable. Compare the results to those achieved with the vertical kernel.

</div>

In [ ]:
cells_gray = cells_gray.astype(float)

vertical_kernel = np.array([
    [-1],
    [ 0],
    [ 1],
])

horizontal_kernel = ...

gradient_vertical = ndi.convolve(cells_gray,vertical_kernel)
gradient_horizontal = ndi.convolve(cells_gray,horizontal_kernel)

fig, ax = plt.subplots(1,2,figsize=(10,5))
ax[0].axis('off')
ax[1].axis('off')
ax[0].imshow(gradient_vertical,cmap='gray')
ax[1].imshow(gradient_horizontal,cmap='gray')

plt.show()

**What if we want to find all of the edges with our filter?**

Thankfully, there's a commonly used filter, the **Sobel** edge filter, that does this for us. First, let's appreciate what it looks like for our bright square.

In [ ]:
plt.imshow(filters.sobel(bright_square))
plt.colorbar()
plt.show()

The sobel filter works like the horizonal and vertical edge detecting kernels we defined above, but it incorporates Gaussian-like smoothing in the direction perpendicular to the direction of edge detection. To create our filter, we'll take the outer product of the smoothing kernel $\begin{bmatrix} 1 & 2 & 1 \end{bmatrix}$ and our edge detection kernel $\begin{bmatrix} -1 & 0 & 1 \end{bmatrix}$.
$$ G_x =\begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \otimes \begin{bmatrix} -1 & 0 & 1 \end{bmatrix} 
= \begin{bmatrix} -1 & 0 & 1\\ -2 & 0 & 2 \\-1 & 0 & 1\end{bmatrix}$$

$$ G_y =\begin{bmatrix} -1 \\ 0 \\ 1 \end{bmatrix} \otimes \begin{bmatrix} 1 & 2 & 1 \end{bmatrix} 
= \begin{bmatrix} -1 & -2 & -1\\ 0 & 0 & 0 \\1 & 2 & 1\end{bmatrix}$$

To detect all edges in out image, we'll convolve our image with $G_x$ and $G_y$ separately and add their results.

Let's start by creating $G_x$ and $G_y$ below using the NumPy function <code>np.outer()</code> to perform the outer product

In [ ]:
# 1D smoothing vector (approximates Gaussian)
smooth = np.array([1, 2, 1])
# 1D derivative vector
deriv = np.array([-1, 0, 1])

# Create 2D Sobel filters using outer product
Gx = np.outer(..., ...)  # horizontal edges
Gy = np.outer(..., ...)  # vertical edges

print("Gx:\n", Gx)
print("Gy:\n", Gy)

Now, let's apply it to our cells!
<div class="alert alert-success">
    
**Task**: Create a figure where the left subplot is a filter of our raw cell image. On the right subplot, apply it to the Gaussian filtered image.
    
</div>

In [ ]:
horizontal_edges = ndi.convolve(cells_gray, Gx)
vertical_edges = ndi.convolve(cells_gray, Gy)
all_edges = np.abs(horizontal_edges) + np.abs(vertical_edges)
raw_sobel = filters.sobel(cells_gray)

fig, ax = plt.subplots(2,2, figsize=(10,8))
fig.suptitle("Sobel Filter on Raw Image")

ax[0,0].imshow(horizontal_edges, cmap='gray')
ax[0,0].set_title("Imaged Convolved with $G_x$")
ax[0,0].axis('off')

ax[0,1].imshow(vertical_edges, cmap='gray')
ax[0,1].set_title("Imaged Convolved with $G_y$")
ax[0,1].axis('off')

ax[1,0].imshow(all_edges, cmap='gray')
ax[1,0].set_title("Computed Sobel Filter")
ax[1,0].axis('off')


ax[1,1].imshow(raw_sobel, cmap='gray')
ax[1,1].set_title("Scipy Sobel Filter")
ax[1,1].axis('off')

fig.tight_layout()
plt.show()

We can also apply the Sobel filter to an image we've smoothed with a Gaussian filter!

In [ ]:
# Create your figure here
gaussian_horizontal_edges = ndi.convolve(gaussian_cells, Gx)
gaussian_vertical_edges = ndi.convolve(gaussian_cells, Gy)
gaussian_edges = np.abs(gaussian_horizontal_edges) + np.abs(gaussian_vertical_edges)
gaussian_sobel = filters.sobel(gaussian_cells)

fig, ax = plt.subplots(2,2, figsize=(10,8))
fig.suptitle("Sobel Filter on Gaussian Image")

ax[0,0].imshow(gaussian_horizontal_edges, cmap='gray')
ax[0,0].set_title("Imaged Convolved with $G_x$")
ax[0,0].axis('off')

ax[0,1].imshow(gaussian_vertical_edges, cmap='gray')
ax[0,1].set_title("Imaged Convolved with $G_y$")
ax[0,0].axis('off')

ax[1,0].imshow(gaussian_edges, cmap='gray')
ax[1,0].set_title("Computed Sobel Filter")
ax[0,0].axis('off')

ax[1,1].imshow(gaussian_sobel, cmap='gray')
ax[1,1].set_title("Scipy Sobel Filter")
ax[0,0].axis('off')

fig.tight_layout()
plt.show()

<hr>

## About this notebook
Some of the code in this notebook was adapted from [this tutorial](https://raphaelvallat.com/bandpower.html) by Raphael Vallat, [these tutorials](https://github.com/voytekresearch/Tutorials) from Torben Noto, _Neural Data Science_ by Pascal Wallisch, and [this tutorial](https://jni.github.io/i2k-skimage-napari/lectures/1_image_filters.html#local-filtering).